This is a file for easy reading and inspect the waveform from established simulation sets

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from NuRadioReco.utilities import units # we use the unit system provided by NuRadioReco
from NuRadioReco.modules.io import eventReader
from NuRadioReco.framework.parameters import particleParameters as pap
from NuRadioReco.framework.parameters import showerParameters as shp



In [ ]:
reader = eventReader.eventReader()
filepath = ''

reader.begin(filepath)
header = reader.get_header()
# event = next(reader.run())
event_num_array = []


for i, event in enumerate(reader.run()):
    primary = event.get_primary()
    if primary is not None:
        print(f"Primary energy: {primary[pap.energy] / units.eV:.3e} eV")

    # Optional: shower energy (if available)
    sim_shower = event.get_first_sim_shower()
    if sim_shower is not None:
        print(f"Shower energy: {sim_shower[shp.energy] / units.eV:.3e} eV")

    print(f"Event with run number {event.get_run_number()} and event_id {event.get_id()}")
    event_num_array.append(event.get_run_number())
    station = event.get_station()

    trigger = station.get_first_trigger()
    # trigger = station.get_triggers()

    # Get a channel from the station
    channel = station.get_channel(1)

    # Get the sampling rate (returns in units of Hz by default)
    sample_rate = channel.get_sampling_rate()

    # Convert to GHz for readability
    print(f"Sample rate: {sample_rate / units.GHz} GHz")


    # Let's plot the traces of the first four channels (=the phased array):
    channel_ids = station.get_channel_ids()[:4]

    fig, axs = plt.subplots(1, 2, figsize=(12,4))

    for channel_id in channel_ids:
        channel = station.get_channel(channel_id)
        axs[0].plot(
            channel.get_times()-channel.get_times()[0] / units.ns,
            channel.get_trace() / units.mV, # unit conversion is easy in NuRadioMC!
            # label=f'Ch. {channel_id}'
        )

        axs[1].plot(
            channel.get_frequencies() / units.MHz,
            abs(channel.get_frequency_spectrum()) / (units.V / units.GHz),
            label=f'Ch. {channel_id}'
        )

    # We'll also plot the trigger time:
    trigger_time = trigger.get_trigger_time()-station.get_channel(channel_ids[0]).get_times()[0]
    trigger_name = trigger.get_name()
    print(trigger_time)
    axs[0].axvline(trigger_time, ls=':', color='grey', label=trigger_name)

    axs[0].set_xlim(-50,500)
    axs[0].legend()
    axs[1].legend()
    axs[0].set_xlabel('t [ns]')
    axs[1].set_xlabel('f [MHz]')
    axs[0].set_ylabel('voltage [mV]')
    axs[1].set_ylabel('amplitude [V/GHz]')
    axs[0].set_title(f"Event trigger at station{event.get_station_ids()}")
    plt.show()

print(event_num_array)

In [ ]:
# Let's plot the traces of the first four channels:
channel_ids = station.get_channel_ids()[:4]

fig, axs = plt.subplots(1, 2, figsize=(12,4))

for channel_id in channel_ids:
    channel = station.get_channel(channel_id)
    axs[0].plot(
        channel.get_times() / units.ns,
        channel.get_trace() / units.mV, # unit conversion is easy in NuRadioMC!
        # label=f'Ch. {channel_id}'
    )

    # plot the frequency spectra:
    axs[1].plot(
        channel.get_frequencies() / units.MHz,
        abs(channel.get_frequency_spectrum()) / (units.V / units.GHz),
        label=f'Ch. {channel_id}'
    )

# We'll also plot the trigger time:
trigger_time = trigger.get_trigger_time()
trigger_name = trigger.get_name()
axs[0].axvline(trigger_time, ls=':', color='grey', label=trigger_name)


axs[0].legend()
axs[1].legend()
axs[0].set_xlabel('t [ns]')
axs[1].set_xlabel('f [MHz]')
axs[0].set_ylabel('voltage [mV]')
axs[1].set_ylabel('amplitude [V/GHz]')

plt.show()